# UTS graf sains data - tugas pendahuluan


Pada nomor 1.1, pembuatan myenv dan instalasi dependensi sudah dilakukan lewat terminal, lalu versinya diperiksa kembali di notebook. Nomor 1.2 menunjukkan koneksi Python ke Neo4j dan relasi pertemanan KNOWS dari Arthur ke tiga temannya. Nomor 1.3 mendokumentasikan dataset events.csv, kode impor batch, dan query pengecekan data RetailRocket yang sudah masuk.

## 1.1 persiapan dan perkenalan lingkungan

Environment dibuat agar paket yang dipakai untuk praktikum tidak bercampur dengan paket proyek lain. Pada pengerjaan ini digunakan **Windows, Python 3.11.9 (versi python di laptop saya), dan virtual environment myenv**. 

Perintah yang sebelumnya dikerjakan di terminal VS Code (PowerShell):

```powershell
python --version
python -m venv myenv
.\myenv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install neo4j pandas tqdm ipykernel
python -m ipykernel install --user --name myenv --display-name "Python (myenv)"
```

Kalau ada environment conda yang sedang aktif, jalankan conda deactivate lebih dulu. Perintah source myvenv/bin/activate dari soal berlaku untuk Linux/macOS, sedangkan Windows menggunakan folder Scripts.

Soal juga mencantumkan daftar paket berikut:

```text
neo4j>=4.0,<4.3
imdbpy==2020.9.25
nose==1.3.7
cinemagoerng>=0.4
imdbinfo>=0.5
imdbpy>=2020.9.25
neo4j==4.2.1
```

Daftar tersebut memuat ketentuan driver Neo4j lama, sementara contoh execute_query() pada bagian 1.2 menggunakan API yang lebih baru. Karena notebook ini menggunakan Python 3.11.9 dan dataset RetailRocket, kode memakai driver Neo4j yang sudah terpasang serta metode session.run(). Paket dari soal tetap dicatat, tetapi tidak dipaksa dipasang seluruhnya karena beberapa tidak dibutuhkan untuk RetailRocket dan kompatibilitasnya perlu diperiksa.

In [1]:
import sys
import importlib.metadata as metadata

print("python:", sys.version.split()[0])
print("executable:", sys.executable)

for paket in ["neo4j", "pandas", "tqdm", "ipykernel"]:
    try:
        print(f"{paket}: {metadata.version(paket)}")
    except metadata.PackageNotFoundError:
        print(f"{paket}: belum terpasang")

python: 3.11.9
executable: d:\SAINS DATA\SEMESTER 7\GRAF SD\uts grafsada\myenv\Scripts\python.exe
neo4j: 6.4.1
pandas: 2.3.3
tqdm: 4.70.1
ipykernel: 7.4.0


Cell sebelumnya menampilkan versi Python dan paket yang terdeteksi. Kalau executable mengarah ke folder myenv, berarti notebook memakai environment yang sesuai. Proses pembuatan dan aktivasi environment tetap didokumentasikan dari terminal karena perintah tersebut harus dijalankan sebelum notebook menggunakan kernel.

## 1.2 mempelajari penggunaan driver neo4j

### koneksi ke database

Tutorial menggunakan database neo4j pada instance lokal melalui neo4j://localhost:7687. Kata sandi diminta saat notebook berjalan agar tidak disimpan langsung di file.

In [2]:
from neo4j import GraphDatabase
from getpass import getpass

uri = "neo4j://127.0.0.1:7687"
username = "neo4j"
password = getpass("password neo4j: ")

driver = GraphDatabase.driver(uri, auth=(username, password))
driver.verify_connectivity()
print("koneksi neo4j berhasil")

koneksi neo4j berhasil


### membuat relasi pertemanan

MERGE digunakan untuk membuat node Person jika belum ada, lalu menghubungkannya dengan relasi KNOWS. Jika cell dijalankan ulang, relasi yang sama tidak dibuat dua kali. Penggunaan session.run() memiliki fungsi yang sama untuk menjalankan Cypher walaupun berbeda dari contoh execute_query() pada soal.

In [3]:
def add_friend(driver, name, friend_name):
    query = """
    MERGE (a:Person {name: $name})
    MERGE (friend:Person {name: $friend_name})
    MERGE (a)-[:KNOWS]->(friend)
    """
    with driver.session(database="neo4j") as session:
        session.run(query, name=name, friend_name=friend_name).consume()


def print_friends(driver, name):
    query = """
    MATCH (a:Person)-[:KNOWS]->(friend:Person)
    WHERE a.name = $name
    RETURN friend.name AS friend_name
    ORDER BY friend_name
    """
    with driver.session(database="neo4j") as session:
        return [row["friend_name"] for row in session.run(query, name=name)]

for friend_name in ["Guinevere", "Lancelot", "Merlin"]:
    add_friend(driver, "Arthur", friend_name)

print("teman Arthur:")
for friend_name in print_friends(driver, "Arthur"):
    print("-", friend_name)

teman Arthur:
- Guinevere
- Lancelot
- Merlin


Query untuk menampilkan graf di Neo4j (jalankan terpisah di Neo4j Query, kemudian simpan screenshot) 

MATCH (a:Person {name: 'Arthur'})-[r:KNOWS]->(b:Person)
RETURN a, r, b;


### jawaban pertanyaan pendahuluan (checkpoint)

**1. Apa nama database yang digunakan pada tutorial ini?**

Nama database yang digunakan adalah neo4j. Nama tersebut ditentukan pada parameter database="neo4j", bukan dari nama instance di Neo4j Desktop.

**2. Sebutkan akses lain (seperti sintaks DDL/DML Cypher) yang dapat digunakan mengakses database lain dalam satu instance.**

Pada Neo4j yang mendukung beberapa database, daftar database dapat dilihat melalui SHOW DATABASES dengan hak akses yang sesuai. Untuk memilih database pada Neo4j Browser, gunakan :use nama_database (perintah khusus Browser, bukan Cypher), sedangkan dari Python bisa melalui driver.session(database="nama_database"). Perintah CREATE DATABASE nama_database merupakan perintah administrasi dan bergantung pada edisi serta hak akses Neo4j. Pada Neo4j Community, dukungan beberapa database dapat terbatas. Setelah database dipilih, operasi data dapat dilakukan menggunakan Cypher seperti CREATE, MERGE, MATCH, SET, dan DELETE.

Contoh yang dapat dicoba di Neo4j Query:
SHOW DATABASES;


MATCH (n)
RETURN labels(n) AS label, count(*) AS jumlah
LIMIT 10;


**3. Apa jenis graf pertemanan yang telah dibuat di sini?**

Graf yang dibuat adalah graf berarah (directed graph). Semua node berlabel Person, sedangkan relasi KNOWS mengarah dari Arthur ke Guinevere, Lancelot, dan Merlin. Relasinya tidak otomatis berlaku dua arah karena hanya dibuat (a)-[:KNOWS]->(friend).


## 1.3 impor data retailrocket ke neo4j

Dataset RetailRocket bersumber dari Kaggle (RetailRocket, sumber yang dicantumkan pada soal: Roman Zykov) dan menggunakan file events.csv.

Untuk mengunduh dataset melalui terminal Windows PowerShell, perintah pada soal adalah:
curl.exe -L -o ecommerce-dataset.zip "https://www.kaggle.com/api/v1/datasets/download/retailrocket/ecommerce-dataset"


File zip kemudian diekstrak dan events.csv diletakkan di folder data/ (atau di folder notebook). Pada praktik sebelumnya dataset sudah diunduh dan diimpor, sehingga bagian ini lebih berfungsi untuk dokumentasi dan pengecekan.

Kolom pada events.csv:

- timestamp - waktu interaksi dalam unix epoch milidetik
- visitorid - id pengunjung
- event - jenis interaksi (view, addtocart, transaction)
- itemid - id produk
- transactionid - id transaksi, biasanya kosong pada event selain transaction

In [4]:
from pathlib import Path
import pandas as pd

lokasi_csv = [Path("data/events.csv"), Path("events.csv")]
file_events = next((p for p in lokasi_csv if p.is_file()), None)

if file_events is None:
    print("events.csv belum ditemukan di data/ atau folder notebook")
    print("sesuaikan lokasi file jika berada di folder lain")
else:
    contoh = pd.read_csv(file_events, nrows=5)
    print("lokasi:", file_events.resolve())
    print("kolom:", contoh.columns.tolist())
    display(contoh)

lokasi: D:\SAINS DATA\SEMESTER 7\GRAF SD\uts grafsada\data\events.csv
kolom: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


### mengecek data yang sudah masuk

Sebelum impor, data yang ada dicek terlebih dahulu. Pada pengerjaan sebelumnya, RetailRocket sudah berhasil dimasukkan ke database neo4j, jadi impor ulang tidak diperlukan. Query di bawah digunakan untuk mengecek node Visitor, node item, dan jenis relationship yang tersedia.

Hasil pengerjaan sebelumnya mencatat 1.407.580 Visitor, 235.061 Item, serta 2.664.312 VIEWED, 69.332 ADDED_TO_CART, dan 22.457 BOUGHT. Angka di output cell berikut perlu dibandingkan dengan catatan tersebut.

In [5]:
query_cek = """
MATCH (v:Visitor)
RETURN count(v) AS jumlah_visitor
"""
query_item = "MATCH (i:Item) RETURN count(i) AS jumlah_item"
query_rel = """
MATCH (:Visitor)-[r:VIEWED|ADDED_TO_CART|BOUGHT]->(:Item)
RETURN type(r) AS jenis, count(r) AS jumlah
ORDER BY jenis
"""

with driver.session(database="neo4j") as session:
    jumlah_visitor = session.run(query_cek).single()["jumlah_visitor"]
    jumlah_item = session.run(query_item).single()["jumlah_item"]
    jumlah_relasi = [(r["jenis"], r["jumlah"]) for r in session.run(query_rel)]

print("jumlah visitor:", jumlah_visitor)
print("jumlah item:", jumlah_item)
print("relationship:")
for jenis, jumlah in jumlah_relasi:
    print(f"- {jenis}: {jumlah}")

jumlah visitor: 1407580
jumlah item: 235061
relationship:
- ADDED_TO_CART: 69332
- BOUGHT: 22457
- VIEWED: 2664312


### mekanisme impor batch

Impor dilakukan per batch menggunakan UNWIND agar data CSV tidak harus dikirim sekaligus. MERGE dipakai untuk memastikan satu pengunjung dan satu produk tidak dibuat berulang, sedangkan CREATE dipakai pada relationship karena setiap baris CSV merupakan satu kejadian interaksi. Setiap relationship menyimpan timestamp, dan transaksi juga menyimpan transactionId.

**Penting:** pada database yang sudah berisi RetailRocket, jangan menjalankan ulang proses impor. Cell berikut hanya mendefinisikan fungsi. Proses impor baru berjalan jika jalankan_impor = True, dan akan ditolak jika sudah ada node Visitor atau Item. Pada driver/server Neo4j lama, sintaks pembuatan constraint mungkin perlu disesuaikan. Karena data sudah terimpor, fungsi constraint tidak perlu dijalankan.

In [6]:
from tqdm.auto import tqdm


def create_constraints(session):
    session.run(
        "CREATE CONSTRAINT visitor_id_unique IF NOT EXISTS "
        "FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE"
    ).consume()
    session.run(
        "CREATE CONSTRAINT item_id_unique IF NOT EXISTS "
        "FOR (i:Item) REQUIRE i.itemId IS UNIQUE"
    ).consume()


def import_batch(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})
    WITH v, i, row
    FOREACH (_ IN CASE WHEN row.event = 'view' THEN [1] ELSE [] END |
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)
    )
    FOREACH (_ IN CASE WHEN row.event = 'addtocart' THEN [1] ELSE [] END |
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)
    )
    FOREACH (_ IN CASE WHEN row.event = 'transaction' THEN [1] ELSE [] END |
        CREATE (v)-[:BOUGHT {
            timestamp: row.timestamp,
            transactionId: row.transactionid
        }]->(i)
    )
    """
    tx.run(query, batch=batch).consume()


def impor_retailrocket(driver, lokasi, batch_size=1000):
    if not Path(lokasi).is_file():
        raise FileNotFoundError(f"file tidak ditemukan: {lokasi}")
    with driver.session(database="neo4j") as session:
        cek = session.run("""
            MATCH (n) WHERE n:Visitor OR n:Item
            RETURN count(n) AS jumlah
        """).single()["jumlah"]
        if cek:
            raise RuntimeError(
                "database sudah memiliki Visitor/Item, impor dibatalkan agar tidak duplikat"
            )
        create_constraints(session)
        total = 0
        for chunk in tqdm(pd.read_csv(lokasi, chunksize=batch_size), desc="impor batch"):
            chunk["timestamp"] = pd.to_numeric(chunk["timestamp"], errors="raise").astype("int64")
            chunk["visitorid"] = pd.to_numeric(chunk["visitorid"], errors="raise").astype("int64")
            chunk["itemid"] = pd.to_numeric(chunk["itemid"], errors="raise").astype("int64")
            chunk["transactionid"] = chunk["transactionid"].fillna("").astype(str)
            records = chunk.to_dict(orient="records")
            if hasattr(session, "execute_write"):
                session.execute_write(import_batch, records)
            else:
                session.write_transaction(import_batch, records)
            total += len(records)
    return total

d:\SAINS DATA\SEMESTER 7\GRAF SD\uts grafsada\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
jalankan_impor = False

if jalankan_impor:
    if file_events is None:
        raise FileNotFoundError("events.csv belum ditemukan")
    total = impor_retailrocket(driver, file_events, batch_size=1000)
    print("baris yang diimpor:", total)
else:
    print("impor dilewati - untuk menghindari duplikasi data RetailRocket")

impor dilewati - untuk menghindari duplikasi data RetailRocket


### query pengecekan graf di neo4j

Query berikut dapat dijalankan di Neo4j Query.


MATCH (v:Visitor)
RETURN count(v) AS jumlah_visitor;



MATCH (i:Item)
RETURN count(i) AS jumlah_item;



MATCH (:Visitor)-[r:VIEWED|ADDED_TO_CART|BOUGHT]->(:Item)
RETURN type(r) AS jenis_relasi, count(r) AS jumlah
ORDER BY jenis_relasi;



MATCH (v:Visitor)-[r:VIEWED|ADDED_TO_CART|BOUGHT]->(i:Item)
RETURN v, r, i
LIMIT 20;


Pada proses impor, satu baris events.csv diterjemahkan menjadi hubungan antara Visitor dan Item. Jenis hubungan mengikuti kolom event, sehingga data view, addtocart, dan transaction memiliki tipe relationship yang berbeda. Impor batch dipakai untuk membatasi jumlah data yang dikirim dalam satu transaksi.